In [5]:
import pandas as pd
import numpy as np
import joblib
import time
import os
from datetime import datetime, timedelta
from flask import Flask, request, jsonify

# Data Science Stack
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier

# SHAP is used for interpretability
try:
    import shap
except ImportError:
    shap = None

# ---------------------------------------------------------
# STEP 0: DATA LOADING (AI4I 2020 Dataset)
# ---------------------------------------------------------
def load_and_prep_dataset(filepath='ai4i2020.csv'):
    """
    Loads the AI4I 2020 Predictive Maintenance Dataset and maps columns.
    Expected source: https://archive.ics.uci.edu/ml/datasets/ai4i+2020+predictive+maintenance+dataset
    """
    if not os.path.exists(filepath):
        print(f"Error: {filepath} not found. Please download it from Kaggle/UCI and place it in the project folder.")
        print("Falling back to synthetic data generation for demonstration purposes...")
        return generate_synthetic_fallback()

    print(f"Step 0: Loading {filepath}...")
    df = pd.read_csv(filepath)

    # Mapping AI4I columns to our processing names
    # AI4I has: Air temperature [K], Process temperature [K], Rotational speed [rpm], Torque [Nm], Tool wear [min]
    column_mapping = {
        'Air temperature [K]': 'temperature',
        'Rotational speed [rpm]': 'vibration', # Rotational speed is a common proxy for vibration in these tasks
        'Torque [Nm]': 'pressure',             # Torque acts as a pressure/stress proxy
        'Machine failure': 'failure',
        'UDI': 'id'
    }
    df = df.rename(columns=column_mapping)
    
    # AI4I is a snapshot dataset. To apply Week 1 "Rolling Windows", 
    # we simulate a temporal sequence based on the UDI (Unique Device Identifier).
    df['timestamp'] = [datetime.now() + timedelta(hours=i) for i in range(len(df))]
    df['machine_id'] = 'ROBOT_001' # Treating the whole set as a single stream for temporal demo
    
    return df

def generate_synthetic_fallback():
    """Generates synthetic data if the CSV is missing so the code doesn't crash."""
    np.random.seed(42)
    data_list = []
    for i in range(2000):
        data_list.append({
            'timestamp': datetime.now() + timedelta(hours=i),
            'machine_id': 'ROBOT_001',
            'temperature': np.random.normal(300, 2),
            'vibration': np.random.normal(1500, 50),
            'pressure': np.random.normal(40, 5),
            'failure': 1 if np.random.random() < 0.02 else 0
        })
    return pd.DataFrame(data_list)

# ---------------------------------------------------------
# WEEK 1: DATA ENGINEERING & TEMPORAL FEATURES
# ---------------------------------------------------------
def week_1_data_engineering(df):
    print("\n--- Week 1: Data Engineering ---")
    df = df.sort_values(['machine_id', 'timestamp'])
    
    # 1. Rolling Window Statistics (Essential for capturing drift)
    # We calculate how sensors behave over the last few readings
    for attr in ['vibration', 'temperature', 'pressure']:
        # Mean over last 4 readings
        df[f'{attr}_roll_mean_4'] = df.groupby('machine_id')[attr].transform(lambda x: x.rolling(window=4).mean())
        # Exponential Moving Average (EMA) - captures sudden shifts
        df[f'{attr}_ema_8'] = df.groupby('machine_id')[attr].transform(lambda x: x.ewm(span=8).mean())
        # Lag Features (Previous state)
        df[f'{attr}_lag_1'] = df.groupby('machine_id')[attr].shift(1)

    # 2. Target Lead Time Logic
    # In the AI4I dataset, 'failure' is the current state.
    # To satisfy the project requirement of "24 hour lead time", we shift the target.
    df['target'] = df.groupby('machine_id')['failure'].shift(-24)
    df['target'] = df['target'].fillna(0).astype(int)
    
    df = df.dropna()
    print(f"Features engineered for AI4I dataset. New shape: {df.shape}")
    return df

# ---------------------------------------------------------
# WEEK 2: MODELING & IMBALANCE HANDLING
# ---------------------------------------------------------
def week_2_modeling(df):
    print("\n--- Week 2: Modeling (XGBoost) ---")
    
    # Features to use (excluding IDs and raw failure)
    exclude = ['timestamp', 'machine_id', 'failure', 'target', 'id', 'Type', 'Product ID']
    features = [c for c in df.columns if c not in exclude]
    
    X = df[features]
    y = df['target']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Manual Oversampling to handle the rare failure events
    print("Upsampling failure events...")
    train_data = pd.concat([X_train, y_train], axis=1)
    df_maj = train_data[train_data.target == 0]
    df_min = train_data[train_data.target == 1]
    df_min_upsampled = df_min.sample(len(df_maj), replace=True, random_state=42)
    df_upsampled = pd.concat([df_maj, df_min_upsampled])
    
    X_res = df_upsampled.drop('target', axis=1)
    y_res = df_upsampled['target']
    
    # XGBoost Optimization
    xgb = XGBClassifier(eval_metric='logloss', random_state=42)
    param_dist = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.1]
    }
    
    rs = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=2, scoring='f1', cv=3)
    rs.fit(X_res, y_res)
    
    best_model = rs.best_estimator_
    print("\nFinal Performance on AI4I Data:")
    print(classification_report(y_test, best_model.predict(X_test)))
    
    return best_model, X_test, features

# ---------------------------------------------------------
# WEEK 3 & 4: XAI AND DEPLOYMENT
# ---------------------------------------------------------
def week_3_xai(model, X_test, features):
    print("\n--- Week 3: Interpretability (SHAP) ---")
    if shap is None:
        print("SHAP not installed. Skipping.")
        return
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test.iloc[:50])
    print("SHAP analysis complete. Decision boundaries identified.")

def week_4_save(model, features):
    print("\n--- Week 4: Deployment ---")
    joblib.dump(model, 'factory_guard_model.pkl')
    joblib.dump(features, 'model_features.pkl')
    print("FactoryGuard AI model saved as .pkl")

# ---------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    # 1. Load the AI4I Dataset
    # Download from: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020
    raw_df = load_and_prep_dataset('ai4i2020.csv')
    
    # 2. Process
    processed_df = week_1_data_engineering(raw_df)
    
    # 3. Trainimport pandas as pd
import numpy as np
import joblib
import time
import os
import re
from datetime import datetime, timedelta
from flask import Flask, request, jsonify

# Data Science Stack
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier

# SHAP is used for interpretability
try:
    import shap
except ImportError:
    shap = None

# ---------------------------------------------------------
# STEP 0: DATA LOADING (AI4I 2020 Dataset)
# ---------------------------------------------------------
def load_and_prep_dataset(filepath='ai4i2020.csv'):
    """
    Loads the AI4I 2020 Predictive Maintenance Dataset and maps columns.
    Expected source: https://archive.ics.uci.edu/ml/datasets/ai4i+2020+predictive+maintenance+dataset
    """
    if not os.path.exists(filepath):
        print(f"Error: {filepath} not found. Please download it from Kaggle/UCI and place it in the project folder.")
        print("Falling back to synthetic data generation for demonstration purposes...")
        return generate_synthetic_fallback()

    print(f"Step 0: Loading {filepath}...")
    df = pd.read_csv(filepath)

    # Mapping AI4I columns to our processing names
    # AI4I has: Air temperature [K], Process temperature [K], Rotational speed [rpm], Torque [Nm], Tool wear [min]
    column_mapping = {
        'Air temperature [K]': 'temperature',
        'Rotational speed [rpm]': 'vibration', # Rotational speed is a common proxy for vibration in these tasks
        'Torque [Nm]': 'pressure',             # Torque acts as a pressure/stress proxy
        'Machine failure': 'failure',
        'UDI': 'id'
    }
    df = df.rename(columns=column_mapping)
    
    # AI4I is a snapshot dataset. To apply Week 1 "Rolling Windows", 
    # we simulate a temporal sequence based on the UDI (Unique Device Identifier).
    df['timestamp'] = [datetime.now() + timedelta(hours=i) for i in range(len(df))]
    df['machine_id'] = 'ROBOT_001' # Treating the whole set as a single stream for temporal demo
    
    return df

def generate_synthetic_fallback():
    """Generates synthetic data if the CSV is missing so the code doesn't crash."""
    np.random.seed(42)
    data_list = []
    for i in range(2000):
        data_list.append({
            'timestamp': datetime.now() + timedelta(hours=i),
            'machine_id': 'ROBOT_001',
            'temperature': np.random.normal(300, 2),
            'vibration': np.random.normal(1500, 50),
            'pressure': np.random.normal(40, 5),
            'failure': 1 if np.random.random() < 0.02 else 0
        })
    return pd.DataFrame(data_list)

# ---------------------------------------------------------
# WEEK 1: DATA ENGINEERING & TEMPORAL FEATURES
# ---------------------------------------------------------
def week_1_data_engineering(df):
    print("\n--- Week 1: Data Engineering ---")
    df = df.sort_values(['machine_id', 'timestamp'])
    
    # 1. Rolling Window Statistics (Essential for capturing drift)
    # We calculate how sensors behave over the last few readings
    for attr in ['vibration', 'temperature', 'pressure']:
        # Mean over last 4 readings
        df[f'{attr}_roll_mean_4'] = df.groupby('machine_id')[attr].transform(lambda x: x.rolling(window=4).mean())
        # Exponential Moving Average (EMA) - captures sudden shifts
        df[f'{attr}_ema_8'] = df.groupby('machine_id')[attr].transform(lambda x: x.ewm(span=8).mean())
        # Lag Features (Previous state)
        df[f'{attr}_lag_1'] = df.groupby('machine_id')[attr].shift(1)

    # 2. Target Lead Time Logic
    # In the AI4I dataset, 'failure' is the current state.
    # To satisfy the project requirement of "24 hour lead time", we shift the target.
    df['target'] = df.groupby('machine_id')['failure'].shift(-24)
    df['target'] = df['target'].fillna(0).astype(int)
    
    df = df.dropna()
    print(f"Features engineered for AI4I dataset. New shape: {df.shape}")
    return df

# ---------------------------------------------------------
# WEEK 2: MODELING & IMBALANCE HANDLING
# ---------------------------------------------------------
def week_2_modeling(df):
    print("\n--- Week 2: Modeling (XGBoost) ---")
    
    # Features to use (excluding IDs and raw failure)
    exclude = ['timestamp', 'machine_id', 'failure', 'target', 'id', 'Type', 'Product ID']
    features = [c for c in df.columns if c not in exclude]
    
    X = df[features]
    y = df['target']
    
    # FIX: Clean feature names for XGBoost (removes [, ], <, and >)
    # XGBoost raises a ValueError if feature names contain these characters.
    clean_feature_names = [re.sub(r'[\[\]<>]', '', str(c)) for c in X.columns]
    X.columns = clean_feature_names
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    # Manual Oversampling to handle the rare failure events
    print("Upsampling failure events...")
    train_data = pd.concat([X_train, y_train], axis=1)
    df_maj = train_data[train_data.target == 0]
    df_min = train_data[train_data.target == 1]
    df_min_upsampled = df_min.sample(len(df_maj), replace=True, random_state=42)
    df_upsampled = pd.concat([df_maj, df_min_upsampled])
    
    X_res = df_upsampled.drop('target', axis=1)
    y_res = df_upsampled['target']
    
    # XGBoost Optimization
    xgb = XGBClassifier(eval_metric='logloss', random_state=42)
    param_dist = {
        'n_estimators': [100, 200],
        'max_depth': [3, 6],
        'learning_rate': [0.1]
    }
    
    rs = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=2, scoring='f1', cv=3)
    rs.fit(X_res, y_res)
    
    best_model = rs.best_estimator_
    print("\nFinal Performance on AI4I Data:")
    print(classification_report(y_test, best_model.predict(X_test)))
    
    return best_model, X_test, X.columns.tolist()

# ---------------------------------------------------------
# WEEK 3 & 4: XAI AND DEPLOYMENT
# ---------------------------------------------------------
def week_3_xai(model, X_test, features):
    print("\n--- Week 3: Interpretability (SHAP) ---")
    if shap is None:
        print("SHAP not installed. Skipping.")
        return
    explainer = shap.TreeExplainer(model)
    # Calculate for a small sample to save time
    shap_values = explainer.shap_values(X_test.iloc[:50])
    print("SHAP analysis complete. Decision boundaries identified.")

def week_4_save(model, features):
    print("\n--- Week 4: Deployment ---")
    joblib.dump(model, 'factory_guard_model.pkl')
    joblib.dump(features, 'model_features.pkl')
    print("FactoryGuard AI model saved as .pkl")

# ---------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    # 1. Load the AI4I Dataset
    # Download from: https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020
    raw_df = load_and_prep_dataset('ai4i2020.csv')
    
    # 2. Process
    processed_df = week_1_data_engineering(raw_df)
    
    # 3. Train
    model, X_test, feature_names = week_2_modeling(processed_df)
    
    # 4. Explain & Save
    week_3_xai(model, X_test, feature_names)
    week_4_save(model, feature_names)
    
    print("\n" + "="*40)
    print("FACTORYGUARD AI: PROJECT COMPLETE")
    print("="*40)
    model, X_test, feature_names = week_2_modeling(processed_df)
    
    # 4. Explain & Save
    week_3_xai(model, X_test, feature_names)
    week_4_save(model, feature_names)
    
    print("\n" + "="*40)
    print("FACTORYGUARD AI: PROJECT COMPLETE")
    print("="*40)

Step 0: Loading ai4i2020.csv...

--- Week 1: Data Engineering ---
Features engineered for AI4I dataset. New shape: (9997, 26)
Step 0: Loading ai4i2020.csv...

--- Week 1: Data Engineering ---
Features engineered for AI4I dataset. New shape: (9997, 26)

--- Week 2: Modeling (XGBoost) ---
Upsampling failure events...

Final Performance on AI4I Data:
              precision    recall  f1-score   support

           0       0.98      0.82      0.89      1932
           1       0.07      0.41      0.13        68

    accuracy                           0.80      2000
   macro avg       0.52      0.62      0.51      2000
weighted avg       0.94      0.80      0.86      2000


--- Week 3: Interpretability (SHAP) ---
SHAP not installed. Skipping.

--- Week 4: Deployment ---
FactoryGuard AI model saved as .pkl

FACTORYGUARD AI: PROJECT COMPLETE

--- Week 2: Modeling (XGBoost) ---
Upsampling failure events...

Final Performance on AI4I Data:
              precision    recall  f1-score   support

